# Systems of Linear Equations

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 01.03 |
| Time | ~55 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/04_systems_of_equations.ipynb)

---

## 🎯 Learning Objective

Solve systems of linear equations with Gaussian elimination and LU decomposition, and understand what happens when a system is over- or under-determined.

---

## 📐 Theory

A system of linear equations is compactly written $A\mathbf{x} = \mathbf{b}$. Three cases
matter:

- **Square, well-determined** ($A$ is $n\times n$ and invertible): exactly one solution.
- **Overdetermined** (more equations than unknowns): generally *no* exact solution exists —
  instead we find the $\mathbf{x}$ that comes as close as possible.
- **Underdetermined** (fewer equations than unknowns): infinitely many solutions exist — we
  typically want the smallest one.

### Gaussian elimination

Gaussian elimination solves a square system by using row operations to reduce $A$ to upper
triangular form, then solving by back-substitution — the same manual technique from a first
algebra course, just done systematically. **LU decomposition** ($A = LU$) packages the
elimination steps themselves into a reusable factorization, so solving for many different
right-hand sides $\mathbf{b}$ becomes cheap after the one-time cost of factoring $A$.

### Least squares for overdetermined systems

When no exact solution exists, we minimize the squared residual:

$$\mathbf{x}^\star = \arg\min_{\mathbf{x}} \|A\mathbf{x}-\mathbf{b}\|_2^2$$

Setting the gradient of this objective to zero yields the **normal equations**:

$$A^\top A\, \mathbf{x}^\star = A^\top \mathbf{b} \qquad\Longrightarrow\qquad
\mathbf{x}^\star = (A^\top A)^{-1}A^\top \mathbf{b}$$

This is the exact closed-form solution behind ordinary least-squares linear regression.

---

## 👁️ Visual Intuition

A well-determined $2\times2$ system is the intersection point of two lines. An overdetermined
system (many data points, one line to fit) has no exact intersection — least squares finds the
line minimizing total squared vertical distance to every point.

In [ ]:
# Left: exact intersection of two lines. Right: least-squares best-fit line through noisy points.
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Square system: 2x + y = 5, x - y = 1  -> unique intersection
x_line = np.linspace(-1, 4, 100)
axes[0].plot(x_line, 5 - 2 * x_line, label="2x + y = 5")
axes[0].plot(x_line, x_line - 1, label="x - y = 1")
sol = np.linalg.solve(np.array([[2.0, 1.0], [1.0, -1.0]]), np.array([5.0, 1.0]))
axes[0].scatter(*sol, color="red", zorder=5, label=f"solution {np.round(sol, 2)}")
axes[0].set_title("Well-determined: unique intersection")
axes[0].legend(fontsize=8); axes[0].set_xlim(-1, 4); axes[0].set_ylim(-3, 6)

# Overdetermined system: many noisy points, no exact line fits all
rng = np.random.default_rng(0)
xs = np.linspace(0, 10, 25)
ys = 2.0 * xs + 1.0 + rng.normal(scale=1.5, size=xs.shape)
A = np.stack([xs, np.ones_like(xs)], axis=1)
w, resid, *_ = np.linalg.lstsq(A, ys, rcond=None)
axes[1].scatter(xs, ys, s=20, color="#4C72B0", label="data (no exact fit exists)")
axes[1].plot(xs, A @ w, color="#DD8452", label=f"least-squares fit: y={w[0]:.2f}x+{w[1]:.2f}")
axes[1].set_title("Overdetermined: least-squares best fit")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

Gaussian elimination with back-substitution, implemented directly from the row-reduction
algorithm, then least squares via the normal equations.

In [ ]:
# Gaussian elimination + back-substitution, and least squares via the normal equations
import numpy as np

def gaussian_elimination_solve(A, b):
    A = A.copy().astype(float)
    b = b.copy().astype(float)
    n = len(b)

    # Forward elimination: zero out everything below the diagonal, column by column
    for col in range(n):
        pivot = A[col, col]
        for row in range(col + 1, n):
            factor = A[row, col] / pivot
            A[row, :] -= factor * A[col, :]
            b[row] -= factor * b[col]

    # Back-substitution: solve for x[n-1], then x[n-2], ... using the now-triangular A
    x = np.zeros(n)
    for row in range(n - 1, -1, -1):
        x[row] = (b[row] - A[row, row + 1:] @ x[row + 1:]) / A[row, row]
    return x

A = np.array([[2.0, 1.0, -1.0], [-3.0, -1.0, 2.0], [-2.0, 1.0, 2.0]])
b = np.array([8.0, -11.0, -3.0])

x_scratch = gaussian_elimination_solve(A, b)
x_numpy = np.linalg.solve(A, b)
print("Scratch solution:", np.round(x_scratch, 4))
print("np.linalg.solve :", np.round(x_numpy, 4))
print("Match:", np.allclose(x_scratch, x_numpy))

print("\n--- Least squares via the normal equations ---")
rng = np.random.default_rng(1)
n_samples, n_features = 40, 2
X = rng.standard_normal((n_samples, n_features))
true_w = np.array([1.5, -0.8])
y = X @ true_w + 0.1 * rng.standard_normal(n_samples)

w_normal_eq = np.linalg.inv(X.T @ X) @ X.T @ y
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
print("True weights          :", true_w)
print("Normal-equations fit   :", np.round(w_normal_eq, 4))
print("np.linalg.lstsq fit    :", np.round(w_lstsq, 4))

## 🤖 Why This Matters for AI

Ordinary least-squares linear regression **is** solving an overdetermined system with the
normal equations. This closed-form solution is the simplest possible instance of "training" —
minimizing a loss function over data — and every gradient-based optimizer in later modules is
solving a version of this same problem when a closed form doesn't exist.

In [ ]:
# Fitting a line to noisy data via the closed-form normal equations -- literally "training"
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
n = 50
x = np.linspace(0, 10, n)
y_true_slope, y_true_intercept = 3.0, -2.0
y = y_true_slope * x + y_true_intercept + rng.normal(scale=2.0, size=n)

A = np.stack([x, np.ones(n)], axis=1)     # design matrix: [x_i, 1] per row
w = np.linalg.inv(A.T @ A) @ A.T @ y       # the normal-equations closed-form "training step"

print(f"True line:      y = {y_true_slope}x + {y_true_intercept}")
print(f"Recovered line: y = {w[0]:.3f}x + {w[1]:.3f}")

plt.figure(figsize=(6, 4))
plt.scatter(x, y, s=15, color="#4C72B0", label="noisy data")
plt.plot(x, A @ w, color="#DD8452", label="fit via normal equations")
plt.legend(); plt.title("Linear regression = solving an overdetermined system")
plt.show()

## 🏋️ Exercises

### Warm-up
1. Solve $\begin{cases}x+y=5\\2x-y=1\end{cases}$ by hand with elimination, then verify with `gaussian_elimination_solve`.

### Practice
2. Factor a $4\times4$ matrix with `scipy.linalg.lu`, and use the resulting $L,U$ (and pivot matrix $P$) to solve $A\mathbf{x}=\mathbf{b}$ for three DIFFERENT right-hand sides $\mathbf{b}$, confirming the answer matches `np.linalg.solve` each time.

### Challenge
3. Build an underdetermined system (e.g. a $2\times4$ matrix $A$ and vector $\mathbf{b}$) and find the minimum-norm solution via `np.linalg.pinv(A) @ b`. Verify it satisfies $A\mathbf{x}=\mathbf{b}$, then generate 3 other valid solutions (add any vector from $A$'s null space) and confirm they all have STRICTLY larger norm than the pseudoinverse solution.

---

## 📚 Further Reading
- Strang, *Introduction to Linear Algebra*, Ch. 2 & Ch. 4 (Orthogonality and least squares)
- [scipy.linalg.lu documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.lu.html)

---

*Next notebook: [Determinants and Inverses](05_determinants_and_inverses.ipynb)*